# MILK10k — Fine-tuning (DINOv2-base)

`01_baseline` notebook'unda görüntü modelini hiç eğitmedik: donmuş DINOv2-large özellikleri + Logistic Regression. Bu notebook'ta görüntü modelinin kendisini (DINOv2-base) bizim lezyonlarla eğitiyoruz.

| | 01_baseline (donmuş) | 02_finetune |
|---|---|---|
| Eğitilen kısım | sadece LR (~45 bin parametre) | DINOv2-base'in tamamı + yeni sınıflandırma katmanı (~86 milyon) |
| İki görüntü | hazır vektörler yan yana | aynı model iki görüntüye bakıyor, birlikte eğitiliyor |
| Augmentation | yok | rastgele kırpma, döndürme, aynalama, renk oynatma |

Son sistem üç modelin olasılık ortalaması: **fine-tuned DINOv2-base + donmuş DINOv2-large + tabular LightGBM** (1/3'er). Bu birleşim fold 0 pilotunda seçilip sabitlendi (pilot fold 0: 0.547; aynı fold'da 01'in sistemi 0.517). Test tahmini, leaderboard'da **0.511** alan kurulumla aynı: fine-tuned model **tüm train** ile bir kez daha eğitiliyor, TTA yok (`TEST_MODEL = "full"`, `TTA = False`). 5 fold modelinin ortalaması + 8 görünümlü TTA CV'de daha yüksek (0.560) ama leaderboard'da **0.474** aldı; bu kurulum ve diğer kombinasyonlar `outputs/finetune/variants/` altına ayrı submission olarak yazılıyor (`history.md` #14).

**Çalıştırma:** Run All. İlerleme çubukları yüzde ve kalan süreyi gösterir.
- Süre: fold başına ~6 dk (RTX 5070) / ~30 dk (RTX 4050); tüm train ile son eğitim ~8 dk / ~38 dk.
- Her fold'un ve son modelin ağırlıkları (`fold{k}.pt`, `full.pt`) ve tahminleri (`fold{k}.npz`, `full.npz`) `outputs/cache/finetune/` altına kaydedilir (git'e girmez). Notebook yarıda kesilirse biten fold'lar atlanır; ağırlığı olan ama tahmini eksik fold yeniden eğitilmez, kayıtlı ağırlıktan tahmin edilir.
- Ortak veri/özellik/fold kodu `notebooks/common.py`'de (01 ile aynı).
- Önce `01_baseline.ipynb` çalışmış olmalı: donmuş DINOv2-large özellikleri onun cache'inden okunuyor.
- Eğitim sırasında GPU kullanan oyun vb. açık olmasın; video izlemek sorun değil.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel

pd.set_option("display.max_columns", 50)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA = ROOT / "data"
CACHE = ROOT / "outputs" / "cache"
FT_DIR = CACHE / "finetune"            # fold ağırlıkları (fold{k}.pt), fold tahminleri (fold{k}.npz), referans modeller
OUT = ROOT / "outputs" / "finetune"    # submission ve sonuç tabloları
FT_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "notebooks"))
# 01_baseline ile ortak kod: notebooks/common.py; veri yükleme (DataLoader worker'ları için ayrı dosyada): notebooks/ft_data.py
from common import (CLASSES, N_FOLDS, SEED, frozen_model, lesion_ids, load_labels, load_split, make_folds,
                    oof_proba, tabular_features, tabular_model)
from ft_data import LesionDataset, preprocess

# fine-tuning ayarları
EPOCHS = 10
BATCH = 16             # lezyon (= 32 görüntü)
LR_BACKBONE = 1e-5     # önceden öğrenilenleri bozmamak için çok küçük
LR_HEAD = 1e-3         # yeni sınıflandırma katmanı sıfırdan öğreniyor
NUM_WORKERS = 4
TTA = False            # fine-tuned tahminde 8 görünüm (4 rotasyon × ayna) ortalaması: CV ana sistem satırı + submission.
                       # 0.511'lik kurulumda yok. CV'de +0.011 (4/5 fold) ama leaderboard'da doğrulanmadı (history.md #14)
TEST_MODEL = "full"    # submission'daki fine-tuned kısım: "full" = tüm train ile eğitilmiş tek model (0.511'lik kurulum)
                       #                                   "folds" = 5 fold modelinin ortalaması (TTA ile leaderboard 0.474)
RUN_FINAL = True       # CV'den sonra test submission yazılsın mı ("full" ise ilk seferde tüm train ile son eğitim)
MAX_STEPS = None       # hızlı deneme için epoch başına adım sınırı (None = tam epoch)

## 1. Veri ve fold'lar

`01_baseline` ile aynı fonksiyonlar (`common.py`): lezyon başına tek satır, aynı tabular özellikler ve aynı 5 fold (seed 42). Böylece sonuçlar 01 ile doğrudan kıyaslanabiliyor.

In [2]:
train, test = load_split(DATA, "Training"), load_split(DATA, "Test")
y = load_labels(DATA, train.index)

sites = sorted(train["site"].dropna().unique())
X_tab, X_tab_test = tabular_features(train, sites), tabular_features(test, sites)
folds = make_folds(y)
all_idx = np.arange(len(y))
print(f"train: {len(train)} lezyon | test: {len(test)} lezyon | fold başına ~{len(folds[0][1])} doğrulama lezyonu")

train: 5240 lezyon | test: 479 lezyon | fold başına ~1048 doğrulama lezyonu


## 2. Referans modeller (01'den)

Fine-tuned modelin yanına konacak iki model, `01_baseline`'daki ayarlarla (`common.py`):
- **Tabular:** LightGBM
- **Donmuş görüntü modeli:** DINOv2-large özellikleri + Logistic Regression (C=0.03)

İkisinin out-of-fold ve test (tüm train ile eğitilmiş) olasılıkları ilk çalıştırmada hesaplanıp (~1–2 dk) `outputs/cache/finetune/reference.npz`'ye yazılıyor; sonraki çalıştırmalar oradan okuyor. Lezyonlar, fold'lar veya model ayarları değişirse cache kendiliğinden yenileniyor. DINOv2-large özellik cache'i yeniden üretilirse `reference.npz`'yi silin.

In [3]:
REF_FILE = FT_DIR / "reference.npz"


def save_atomic(path, save):
    """Önce geçici dosyaya yazıp sonra yeniden adlandırır: yarıda kesilen kayıt bitmiş dosya gibi görünmesin."""
    tmp = path.with_name(f"{path.stem}.tmp{path.suffix}")
    save(tmp)
    tmp.replace(path)


def dinov2_large_features(df, split):
    cache = CACHE / f"dinov2_large_{split}.npz"
    assert cache.exists(), "önce 01_baseline.ipynb'yi BACKBONE = 'dinov2_large' ile çalıştır"
    z = np.load(cache)
    assert (z["lesion_id"] == lesion_ids(df)).all()
    return z["X"]


def reference_predictions():
    """LightGBM ve donmuş LR: out-of-fold (p_tab, p_frozen) ve test (p_tab_test, p_frozen_test) olasılıkları."""
    fold_of = np.empty(len(y), int)
    for k, (_, va) in enumerate(folds):
        fold_of[va] = k
    key = {"train_id": lesion_ids(train), "test_id": lesion_ids(test), "y": y,
           "fold_of": fold_of, "config": np.array(repr(tabular_model()) + repr(frozen_model()))}
    if REF_FILE.exists():
        with np.load(REF_FILE) as z:  # with: Windows'ta açık kalan dosya sonra üzerine yazılamıyor
            if all(name in z.files and np.array_equal(z[name], v) for name, v in key.items()):
                print(f"referans modeller cache'ten: {REF_FILE.relative_to(ROOT)}")
                return {name: z[name] for name in ("p_tab", "p_frozen", "p_tab_test", "p_frozen_test")}
        print("referans cache'i eski (lezyon/fold/ayar değişmiş), yeniden hesaplanıyor")

    X_frozen, X_frozen_test = dinov2_large_features(train, "train"), dinov2_large_features(test, "test")
    ref = {"p_tab": oof_proba(tabular_model, X_tab, y, folds),
           "p_frozen": oof_proba(frozen_model, X_frozen, y, folds),
           "p_tab_test": tabular_model().fit(X_tab, y).predict_proba(X_tab_test),
           "p_frozen_test": frozen_model().fit(X_frozen, y).predict_proba(X_frozen_test)}
    save_atomic(REF_FILE, lambda p: np.savez(p, **ref, **key))
    return ref


def f1(idx, p):
    return f1_score(y[idx], p.argmax(1), average="macro", labels=range(len(CLASSES)), zero_division=0)


ref = reference_predictions()
p_tab, p_frozen = ref["p_tab"], ref["p_frozen"]
print(f"tabular {f1(all_idx, p_tab):.4f} | donmuş DINOv2-large {f1(all_idx, p_frozen):.4f} | "
      f"01'in sistemi (ikisinin ortalaması) {f1(all_idx, (p_frozen + p_tab) / 2):.4f}")

tabular 0.3334 | donmuş DINOv2-large 0.4821 | 01'in sistemi (ikisinin ortalaması) 0.4867


## 3. Fine-tuning

**Model:** Klinik ve dermoskopik görüntü **aynı** DINOv2-base'ten geçiyor. Her görüntüden CLS token + patch token ortalaması alınıp iki görüntünün özellikleri yan yana konuyor, oradan 11 sınıfa giden tek bir doğrusal katman var.

**Eğitim:**
- AdamW; backbone için çok küçük öğrenme hızı (1e-5), yeni katman için 1e-3.
- İlk epoch'ta yavaş ısınma (warmup), sonra kosinüs şeklinde azalan öğrenme hızı.
- Sınıf ağırlıklı (balanced) cross-entropy; mixed precision (fp16); gradient clipping.
- **Gradient checkpointing** sadece GPU'da yer azsa açılıyor: hafızayı ~2 GB'a indirir ama ~%20–25 yavaşlatır.
- Her fold'da **son epoch** kullanılıyor. En iyi epoch'u doğrulama fold'una bakarak seçmek skoru şişirirdi. Bu yüzden doğrulama fold'u sadece eğitim bitince bir kez tahmin ediliyor (eskiden her epoch'ta sadece log için tahmin ediliyordu, eğitim süresinin ~%5–10'u).

**Tahmin (TTA):** Ortadan kırpılmış 224 px görüntünün 8 hali (0/90/180/270° döndürme × yatay ayna; dikey ayna da bunların içinde) modelden geçirilip olasılıklar ortalanıyor. Klinik ve dermoskopik görüntüye aynı dönüşüm uygulanıyor. İlk görünüm orijinal görüntü, yani TTA'sız skor da aynı hesaptan çıkıyor. Donmuş özelliklerde 8 görünümlü TTA +0.009 getirmişti (`history.md` #9). Fine-tuned modelde CV'de +0.011 getirdi ama leaderboard'da henüz ayrı ölçülmedi, bu yüzden varsayılan `TTA = False`.

In [4]:
class TwoViewDino(nn.Module):
    """Klinik ve dermoskopik görüntü aynı DINOv2'den geçer; iki görüntünün özellikleri yan yana -> sınıflandırma katmanı."""

    def __init__(self, ckpt):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("facebook/dinov2-base")
        if ckpt:
            self.backbone.gradient_checkpointing_enable()
        d = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(4 * d, len(CLASSES)))

    def embed(self, x):
        out = self.backbone(pixel_values=x)
        return torch.cat([out.pooler_output, out.last_hidden_state[:, 1:].mean(1)], 1)  # CLS + patch ortalaması

    def forward(self, clin, derm):
        e = self.embed(torch.cat([clin, derm]))
        return self.head(torch.cat([e[:len(clin)], e[len(clin):]], 1))


def make_loader(df, idx, labels, train_mode):
    ds = LesionDataset(df["clin"].values[idx], df["derm"].values[idx], labels, train_mode)
    return DataLoader(ds, batch_size=BATCH if train_mode else 32, shuffle=train_mode, drop_last=train_mode,
                      num_workers=NUM_WORKERS, persistent_workers=train_mode, pin_memory=True)


def steps_per_epoch(n_lesions):
    n = n_lesions // BATCH
    return n if MAX_STEPS is None else min(MAX_STEPS, n)


# TTA görünümleri: 4 rotasyon × (aynasız, yatay ayna). İlki (0, False) orijinal görüntü.
VIEWS = [(rot, flip) for flip in (False, True) for rot in range(4)]


def view(x, rot, flip):
    x = torch.flip(x, dims=[3]) if flip else x
    return torch.rot90(x, rot, dims=[2, 3])


@torch.no_grad()
def predict(model, loader):
    """Görünüm başına olasılıklar, şekil (8, lezyon, 11). [0] = TTA'sız (ortadan kırpma), .mean(0) = 8 görünümlü TTA."""
    model.eval()
    probs = []
    for clin, derm, _ in loader:
        clin, derm = clin.cuda(non_blocking=True), derm.cuda(non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            probs.append(torch.stack([torch.softmax(model(view(clin, r, f), view(derm, r, f)).float(), 1)
                                      for r, f in VIEWS]).cpu().numpy())
    return np.concatenate(probs, axis=1)


def train_model(tr_idx, progress, on_epoch=None):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    ckpt = torch.cuda.mem_get_info()[0] / 2**30 < 5.3
    loader = make_loader(train, tr_idx, y[tr_idx], train_mode=True)
    model = TwoViewDino(ckpt).cuda()
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": model.head.parameters(), "lr": LR_HEAD}], weight_decay=0.05)
    n_steps = steps_per_epoch(len(tr_idx))
    total = EPOCHS * n_steps
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / n_steps) * 0.5 * (1 + np.cos(np.pi * min(1.0, s / total))))
    class_w = len(tr_idx) / (len(CLASSES) * np.bincount(y[tr_idx], minlength=len(CLASSES)))
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_w, dtype=torch.float32, device="cuda"))
    scaler = torch.amp.GradScaler()
    progress.write(f"gradient checkpointing: {'açık' if ckpt else 'kapalı'}")

    for epoch in range(EPOCHS):
        model.train()
        t, losses = time.time(), []
        for step, (clin, derm, labels) in enumerate(loader):
            if step == n_steps:
                break
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(clin.cuda(non_blocking=True), derm.cuda(non_blocking=True))
            loss = loss_fn(logits.float(), labels.cuda())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(loss.item())
            progress.update(1)
        if on_epoch:
            on_epoch(model, epoch, float(np.mean(losses)), time.time() - t)
    return model


def weights_file(k):
    """k = 0–4: fold modeli; k = "full": tüm train ile eğitilmiş son model."""
    return FT_DIR / ("full.pt" if k == "full" else f"fold{k}.pt")


# ağırlıklarla birlikte kaydedilir; farklı ayarla (ör. MAX_STEPS'li hızlı deneme) eğitilmiş ağırlıklar yeniden kullanılmaz
TRAIN_CONFIG = {"epochs": EPOCHS, "batch": BATCH, "lr_backbone": LR_BACKBONE, "lr_head": LR_HEAD,
                "max_steps": MAX_STEPS, "seed": SEED}


def save_weights(model, k):
    save_atomic(weights_file(k), lambda p: torch.save({"config": TRAIN_CONFIG, "state_dict": model.state_dict()}, p))


def saved_config(k):
    """fold{k}.pt'nin eğitim ayarları; dosya yoksa None. mmap: 350 MB'ı belleğe okumadan sadece ayarlara bakar."""
    if not weights_file(k).exists():
        return None
    return torch.load(weights_file(k), map_location="cpu", weights_only=True, mmap=True)["config"]


def load_weights(k):
    """Kayıtlı fold modelini yeniden eğitmeden yükler. weights_only=True: dosyadan sadece tensör/sayı/metin okunur, kod çalıştırılmaz."""
    saved = torch.load(weights_file(k), map_location="cuda", weights_only=True)
    assert saved["config"] == TRAIN_CONFIG, f"fold {k} ağırlıkları farklı ayarlarla eğitilmiş: {saved['config']}"
    model = TwoViewDino(ckpt=False).cuda()
    model.load_state_dict(saved["state_dict"])
    return model

### 5-fold eğitim

Her fold için model kalan 4 fold'la eğitiliyor, ağırlıkları `fold{k}.pt` olarak kaydediliyor (~350 MB/fold). Sonra aynı model:
- doğrulama fold'unu ve test setini 8 görünümle tahmin ediyor (`fold{k}.npz`: `va`, `p_va_views`, `p_test_views`, `test_id`),
- fold skorlarını yazıyor: **fine-tuned** (modelin tek başına) ve **üçlü** (fine-tuned + donmuş + tabular), TTA'sız → TTA.

Tahminleri hazır fold'lar atlanıyor. `fold{k}.pt` varsa ama tahmin dosyası yoksa ya da eski sürümdense (TTA'sız eski `fold{k}.npz`) model eğitilmeden yüklenip sadece tahmin ediliyor. `fold{k}.pt` içinde eğitim ayarları (`TRAIN_CONFIG`: EPOCHS, BATCH, LR'ler, MAX_STEPS, SEED) da saklanıyor; farklı ayarlarla eğitilmiş ağırlıklar (ör. `MAX_STEPS = 2` ile hızlı deneme) yeniden kullanılmıyor, fold yeniden eğitiliyor.

In [5]:
preprocess(list(train["clin"]) + list(train["derm"]) + list(test["clin"]) + list(test["derm"]))
TEST_ID = lesion_ids(test)


def fold_file(k):
    return FT_DIR / f"fold{k}.npz"


def fold_done(k):
    """Bu ayarlarla eğitilmiş ağırlıklar ve bu sürümün tahminleri (TTA + test) diskteyse fold bitmiş sayılır."""
    if saved_config(k) != TRAIN_CONFIG or not fold_file(k).exists():
        return False
    with np.load(fold_file(k)) as z:
        return ({"p_va_views", "p_test_views", "test_id"} <= set(z.files) and np.array_equal(z["va"], folds[k][1])
                and np.array_equal(z["test_id"], TEST_ID))


todo = [k for k in range(N_FOLDS) if not fold_done(k)]
to_train = [k for k in todo if saved_config(k) != TRAIN_CONFIG]
stale = [k for k in to_train if weights_file(k).exists()]
print("hazır fold'lar:", [k for k in range(N_FOLDS) if k not in todo], "| tahmin edilecek:", todo, "| eğitilecek:", to_train)
if stale:
    print(f"uyarı: fold {stale} ağırlıkları farklı ayarlarla eğitilmiş ({saved_config(stale[0])}), üzerine yazılacak")
progress = tqdm(total=sum(EPOCHS * steps_per_epoch(len(folds[k][0])) for k in to_train), desc="CV fine-tuning", unit="adım")
te_loader = make_loader(test, np.arange(len(test)), np.zeros(len(test), int), train_mode=False)

for k in todo:
    tr, va = folds[k]
    if k in to_train:
        model = train_model(tr, progress, on_epoch=lambda m, epoch, loss, sec: progress.write(
            f"fold {k} | epoch {epoch + 1:2d}/{EPOCHS} | loss {loss:.3f} | {sec / 60:.1f} dk"))
        save_weights(model, k)
    else:
        model = load_weights(k)
        progress.write(f"fold {k} | kayıtlı ağırlıklar yüklendi: {weights_file(k).relative_to(ROOT)}")

    t = time.time()
    p_va_views = predict(model, make_loader(train, va, y[va], train_mode=False))
    p_test_views = predict(model, te_loader)
    assert p_va_views.shape == (len(VIEWS), len(va), len(CLASSES)) and np.allclose(p_va_views.sum(-1), 1, atol=1e-3)
    assert p_test_views.shape == (len(VIEWS), len(test), len(CLASSES)) and np.allclose(p_test_views.sum(-1), 1, atol=1e-3)
    save_atomic(fold_file(k), lambda p: np.savez(p, va=va, p_va_views=p_va_views, p_test_views=p_test_views, test_id=TEST_ID))

    p1, p8 = p_va_views[0], p_va_views.mean(0)
    trio = lambda p: (p + p_frozen[va] + p_tab[va]) / 3
    progress.write(f"fold {k} | val: fine-tuned {f1(va, p1):.4f} → TTA {f1(va, p8):.4f} | "
                   f"üçlü {f1(va, trio(p1)):.4f} → TTA {f1(va, trio(p8)):.4f} | tahmin {(time.time() - t) / 60:.1f} dk")
    del model
    torch.cuda.empty_cache()
progress.close()

hazır fold'lar: [] | tahmin edilecek: [0, 1, 2, 3, 4] | eğitilecek: [0, 1, 2, 3, 4]
gradient checkpointing: kapalı
fold 0 | epoch  1/10 | loss 2.218 | 0.9 dk
fold 0 | epoch  2/10 | loss 1.807 | 0.6 dk
fold 0 | epoch  3/10 | loss 1.368 | 0.6 dk
fold 0 | epoch  4/10 | loss 1.084 | 0.6 dk
fold 0 | epoch  5/10 | loss 0.793 | 0.6 dk
fold 0 | epoch  6/10 | loss 0.559 | 0.6 dk
fold 0 | epoch  7/10 | loss 0.388 | 0.6 dk
fold 0 | epoch  8/10 | loss 0.295 | 0.6 dk
fold 0 | epoch  9/10 | loss 0.218 | 0.6 dk
fold 0 | epoch 10/10 | loss 0.193 | 0.6 dk
fold 0 | val: fine-tuned 0.5357 → TTA 0.5417 | üçlü 0.5457 → TTA 0.5595 | tahmin 0.9 dk
gradient checkpointing: kapalı
fold 1 | epoch  1/10 | loss 2.234 | 0.8 dk
fold 1 | epoch  2/10 | loss 1.781 | 0.6 dk
fold 1 | epoch  3/10 | loss 1.395 | 0.6 dk
fold 1 | epoch  4/10 | loss 1.072 | 0.6 dk
fold 1 | epoch  5/10 | loss 0.775 | 0.6 dk
fold 1 | epoch  6/10 | loss 0.536 | 0.6 dk
fold 1 | epoch  7/10 | loss 0.375 | 0.6 dk
fold 1 | epoch  8/10 | loss 0.283 |

CV fine-tuning:   0%|          | 0/13100 [00:01<?, ?adım/s]C:\Users\yakup\AppData\Local\Temp\ipykernel_22180\1531486071.py:86: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()
CV fine-tuning:  20%|██        | 2620/13100 [07:14<24:00,  7.28adım/s]C:\Users\yakup\AppData\Local\Temp\ipykernel_22180\1531486071.py:86: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pyt

## 4. Sonuçlar (5-fold CV)

Bütün sistemler aynı fold'larda, `01_baseline`'ın sistemiyle (donmuş + tabular) kıyaslanıyor. Karar **ana sisteme** (fine-tuned + donmuş + tabular; `TTA = True` ise fine-tuned kısmı 8 görünümlü) göre verilecek; diğer satırlar bilgi için. Altındaki satır TTA'nın aynı modeller üzerindeki etkisini ve kayıtlı referansa (`history.md` #13, TTA'sız 0.544) göre farkı gösteriyor.

In [6]:
BENCHMARK = 0.544  # history.md #13: fine-tuned + donmuş + tabular, TTA'sız

p_ft, p_ft_tta = np.zeros_like(p_tab), np.zeros_like(p_tab)
p_test_views = []  # fold başına (8, n_test, 11)
for k in range(N_FOLDS):
    with np.load(fold_file(k)) as z:
        assert (z["va"] == folds[k][1]).all(), f"fold {k} dosyası farklı bir bölünmeye ait"
        assert (z["test_id"] == TEST_ID).all(), f"fold {k} test tahminleri farklı bir test listesine ait"
        p_ft[z["va"]] = z["p_va_views"][0]
        p_ft_tta[z["va"]] = z["p_va_views"].mean(0)
        p_test_views.append(z["p_test_views"])
p_main = p_ft_tta if TTA else p_ft

systems = {
    "01: donmuş + tabular": (p_frozen + p_tab) / 2,
    "fine-tuned": p_ft,
    "fine-tuned (TTA)": p_ft_tta,
    "fine-tuned + tabular": (p_main + p_tab) / 2,
    "fine-tuned + donmuş": (p_main + p_frozen) / 2,
    "fine-tuned + donmuş + tabular (TTA'sız)": (p_ft + p_frozen + p_tab) / 3,
    "fine-tuned + donmuş + tabular (ana sistem)": (p_main + p_frozen + p_tab) / 3,
}


def fold_scores(p):
    return np.array([f1(va, p[va]) for _, va in folds])


ref_folds = fold_scores(systems["01: donmuş + tabular"])
results = pd.DataFrame({name: {"macro_f1": f1(all_idx, p), "fold_std": fold_scores(p).std(),
                               "01'e göre": fold_scores(p).mean() - ref_folds.mean(),
                               "01'i geçtiği fold": int((fold_scores(p) > ref_folds).sum())}
                        for name, p in systems.items()}).T
results.to_csv(OUT / "cv_results.csv")

no_tta = (p_ft + p_frozen + p_tab) / 3
with_tta = (p_ft_tta + p_frozen + p_tab) / 3
gain = fold_scores(with_tta) - fold_scores(no_tta)
print(f"TTA etkisi (üçlü sistem, aynı modeller): {f1(all_idx, no_tta):.4f} → {f1(all_idx, with_tta):.4f} "
      f"(Δ {f1(all_idx, with_tta) - f1(all_idx, no_tta):+.4f}, {int((gain > 0).sum())}/{N_FOLDS} fold'da önde) | "
      f"kayıtlı referans {BENCHMARK} → Δ {f1(all_idx, with_tta) - BENCHMARK:+.4f}")
results.round(4)

TTA etkisi (üçlü sistem, aynı modeller): 0.5492 → 0.5598 (Δ +0.0106, 4/5 fold'da önde) | kayıtlı referans 0.544 → Δ +0.0158


,macro_f1,fold_std,01'e göre,01'i geçtiği fold
01: donmuş + tabular,0.4867,0.0241,0.0000,0.0
fine-tuned,0.5350,0.0248,0.0452,5.0
fine-tuned (TTA),0.5543,0.0347,0.0647,5.0
fine-tuned + tabular,0.5481,0.0210,0.0586,5.0
fine-tuned + donmuş,0.5609,0.0248,0.0711,5.0
fine-tuned + donmuş + tabular (TTA'sız),0.5492,0.0280,0.0576,5.0
fine-tuned + donmuş + tabular (ana sistem),0.5598,0.0290,0.0681,5.0


In [7]:
per_class = pd.DataFrame({name: f1_score(y, p.argmax(1), average=None, labels=range(len(CLASSES)), zero_division=0)
                          for name, p in systems.items()}, index=CLASSES).round(3)
per_class.insert(0, "n", np.bincount(y, minlength=len(CLASSES)))
per_class.to_csv(OUT / "cv_per_class_f1.csv")
per_class

,n,01: donmuş + tabular,fine-tuned,fine-tuned (TTA),fine-tuned + tabular,fine-tuned + donmuş,fine-tuned + donmuş + tabular (TTA'sız),fine-tuned + donmuş + tabular (ana sistem)
AKIEC,303,0.425,0.547,0.551,0.564,0.528,0.516,0.524
BCC,2522,0.858,0.885,0.894,0.892,0.898,0.896,0.898
BEN_OTH,44,0.254,0.209,0.212,0.211,0.246,0.230,0.267
BKL,544,0.451,0.537,0.554,0.547,0.557,0.559,0.553
DF,52,0.506,0.528,0.598,0.571,0.640,0.614,0.621
INF,50,0.152,0.365,0.408,0.386,0.348,0.306,0.353
MAL_OTH,9,0.000,0.000,0.000,0.000,0.000,0.000,0.000
MEL,450,0.591,0.620,0.629,0.634,0.641,0.645,0.647
NV,746,0.774,0.789,0.788,0.807,0.799,0.806,0.814
SCCKA,473,0.652,0.689,0.702,0.693,0.723,0.712,0.715


## 5. Test submission

Varsayılan (`TEST_MODEL = "full"`, `TTA = False`) leaderboard'da **0.511** alan kurulum: fine-tuned model aynı ayarlarla **tüm train** verisiyle (5240 lezyon) bir kez daha eğitilip test tahmin ediliyor; donmuş model ve tabular model de tüm train ile eğitilmiş (`reference.npz`). Üçünün ortalamasında en olası sınıfa 1, diğerlerine 0 yazılıyor. Son modelin ağırlıkları `full.pt`, test tahminleri (8 görünüm) `full.npz`; tekrar çalıştırınca yeniden eğitilmiyor.

`TEST_MODEL = "folds"`: fine-tuned kısım CV'deki 5 fold modelinin test tahminlerinin ortalaması (`TTA = True` ile 40 tahminin ortalaması). Bu kurulum leaderboard'da **0.474** aldı. Ortalama alınca nadir sınıf tahminleri azalıyor (testte BEN_OTH 1 → 0, beş nadir sınıf toplam 23 → 18) ve CV bu kurulumu hiç ölçmüyor: CV'de her lezyonu tek bir fold modeli tahmin ediyor (`history.md` #14).

Leaderboard'da karşılaştırma için bütün kombinasyonlar `outputs/finetune/variants/submission_{full,folds}_{notta,tta}.csv` olarak da yazılıyor.

In [ ]:
def full_model_test_views():
    """Tüm train ile eğitilmiş tek modelin test tahminleri, şekil (8, n_test, 11); [0] = TTA'sız.
    Ağırlıklar full.pt, tahminler full.npz: TTA ayarını değiştirmek yeniden eğitim gerektirmez."""
    pred_file = FT_DIR / "full.npz"
    if saved_config("full") == TRAIN_CONFIG and pred_file.exists():
        with np.load(pred_file) as z:
            if np.array_equal(z["test_id"], TEST_ID):
                return z["p_test_views"]
    if saved_config("full") == TRAIN_CONFIG:
        model = load_weights("full")
        print(f"son model kayıtlı ağırlıklardan yüklendi: {weights_file('full').relative_to(ROOT)}")
    else:
        progress = tqdm(total=EPOCHS * steps_per_epoch(len(all_idx)), desc="son eğitim (tüm train)", unit="adım")
        model = train_model(all_idx, progress, on_epoch=lambda m, epoch, loss, sec: progress.write(
            f"son eğitim | epoch {epoch + 1:2d}/{EPOCHS} | loss {loss:.3f} | {sec / 60:.1f} dk"))
        progress.close()
        save_weights(model, "full")
    views = predict(model, te_loader)
    assert views.shape == (len(VIEWS), len(test), len(CLASSES)) and np.allclose(views.sum(-1), 1, atol=1e-3)
    save_atomic(pred_file, lambda p: np.savez(p, p_test_views=views, test_id=TEST_ID))
    del model
    torch.cuda.empty_cache()
    return views


def ensemble_test(views, tta):
    """fine-tuned (TTA'lı/TTA'sız) + donmuş + tabular, 1/3'er."""
    return ((views.mean(0) if tta else views[0]) + ref["p_frozen_test"] + ref["p_tab_test"]) / 3


def write_submission(p_test, path):
    # ilk sütun 'lesion_id' olmalı: yarışma sayfası 'lesion' diyor ama skorlama kodu 'lesion_id' istiyor
    sub = pd.DataFrame(np.eye(len(CLASSES))[p_test.argmax(1)], columns=CLASSES, index=test.index).rename_axis("lesion_id")
    assert len(sub) == len(test) and (sub.sum(axis=1) == 1).all()
    sub.to_csv(path)
    return sub


if RUN_FINAL:
    assert TEST_MODEL in ("full", "folds"), TEST_MODEL
    test_views = {"folds": np.stack(p_test_views).mean(0)}  # (8, n_test, 11): 5 fold modelinin ortalaması
    if TEST_MODEL == "full" or weights_file("full").exists():
        test_views["full"] = full_model_test_views()

    p_test = ensemble_test(test_views[TEST_MODEL], TTA)
    assert p_test.shape == (len(test), len(CLASSES)) and np.allclose(p_test.sum(1), 1, atol=1e-3)
    sub = write_submission(p_test, OUT / "submission.csv")
    pd.DataFrame(p_test, columns=CLASSES, index=test.index).rename_axis("lesion_id").to_csv(OUT / "test_probabilities.csv")
    print(f"yazıldı: {(OUT / 'submission.csv').relative_to(ROOT)}  (TEST_MODEL={TEST_MODEL!r}, TTA={TTA})")

    # leaderboard'da A/B için bütün kombinasyonlar
    VARIANTS = OUT / "variants"
    VARIANTS.mkdir(exist_ok=True)
    counts = {}
    for name, views in test_views.items():
        for tta in (False, True):
            p = ensemble_test(views, tta)
            key = f"{name}_{'tta' if tta else 'notta'}"
            write_submission(p, VARIANTS / f"submission_{key}.csv")
            counts[f"{key} (ana ile aynı: %{(p.argmax(1) == p_test.argmax(1)).mean() * 100:.0f})"] = np.bincount(p.argmax(1), minlength=len(CLASSES))
    print(f"varyantlar: {VARIANTS.relative_to(ROOT)}")

    n = np.bincount(y, minlength=len(CLASSES))
    display(pd.DataFrame({"train gerçek %": n / len(y) * 100, "test tahmin %": sub.sum().values / len(sub) * 100,
                          **{k: v for k, v in counts.items()}}, index=CLASSES).round(1))

## Notlar

- Sonuçları `history.md`'ye yeni bir deney olarak ekle (CV tablosu + varsa leaderboard skoru).
- Gönderilecek dosya: `outputs/finetune/submission.csv` (varsayılan ayarlarla 0.511'lik kurulum). Diğer kombinasyonlar `outputs/finetune/variants/` altında; 0.511 alan orijinal dosyanın birebir kopyası `variants/submission_original_lb0511.csv`. 01'in submission'ı `outputs/baseline/` altında duruyor.
- Fold ağırlıkları `outputs/cache/finetune/fold{k}.pt`, son model `full.pt` (6 × ~350 MB, git'e girmez). Yeni bir TTA/ensemble denemesi için yeniden eğitim gerekmez: `fold{k}.npz` dosyalarını silip notebook'u tekrar çalıştırmak yeterli.
- `TRAIN_CONFIG`'teki ayarlar değişirse fold'lar kendiliğinden yeniden eğitilir. Augmentation (`ft_data.py`) veya model mimarisi değişirse bu otomatik anlaşılmaz: `fold*.pt`, `fold*.npz`, `full.pt` ve `full.npz` elle silinmeli.
- Bu notebook'un fold 0 sonucu (0.544'lük kayıtta) pilot çalıştırmadan geliyor (aynı ayarlar, aynı fold).